# Task 3.2 & 3.3: Supervised Modeling and Evaluation
## Regression (Revenue Forecasting) and Multi-class Classification (Delivery & Segment)

This notebook contains the MLflow-tracked experiments for both the continuous regression and multi-class classification tasks, including cross-validation, hyperparameter tuning, and confusion matrix analysis.

In [ ]:
# Install all required dependencies into the current kernel environment
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn datasets scikit-learn joblib mlflow --quiet
print("All dependencies installed successfully.")

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.linear_model import SGDRegressor, LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, accuracy_score
)

sns.set_theme(style="whitegrid")
mlflow.set_experiment("AuraCart_Supervised_Modeling")
os.makedirs('../artifacts', exist_ok=True)
os.makedirs('../report_screenshots', exist_ok=True)
print("Setup complete.")

## 1. Load and Prepare Data

In [ ]:
print("Loading real AuraCart dataset...")
dataset = load_dataset("millat/e-commerce-orders")
df = pd.DataFrame(dataset['train'])
print(f"Loaded {len(df)} records")

# Engineer features
df['total_value'] = df['price'] * df['quantity']
le_delivery = LabelEncoder()
le_segment = LabelEncoder()
df['delivery_encoded'] = le_delivery.fit_transform(df['delivery_status'])
df['segment_encoded'] = le_segment.fit_transform(df['customer_segment'])

# Build feature matrix (20 features via one-hot encoding)
X_raw = pd.get_dummies(df[['quantity', 'price', 'category', 'payment_method', 'device_type', 'channel']], drop_first=False)
# Pad or trim to exactly 20 features for API compatibility
feature_cols = X_raw.columns.tolist()[:20]
X = X_raw[feature_cols].values

y_regression = df['total_value'].values
y_delivery = df['delivery_encoded'].values
y_segment = df['segment_encoded'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Delivery classes: {le_delivery.classes_}")
print(f"Segment classes: {le_segment.classes_}")

## 2. Task 3.2: Revenue Forecasting (Regression)
We implement a Multiple Linear Regression model using SGDRegressor with MLflow tracking.

In [ ]:
with mlflow.start_run(run_name="Revenue_Forecasting_Regression"):
    X_train, X_test, y_train, y_test = train_test_split(X, y_regression, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    reg_model = SGDRegressor(eta0=0.01, max_iter=1000, random_state=42, tol=1e-3)
    reg_model.fit(X_train_s, y_train)
    y_pred = reg_model.predict(X_test_s)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.sklearn.log_model(reg_model, "regression_model")
    
    print(f"Regression Results -> MSE: {mse:.2f} | MAE: {mae:.2f} | R2: {r2:.4f}")
    joblib.dump(reg_model, '../artifacts/model.joblib')
    print("Model saved to ../artifacts/model.joblib")

## 3. Task 3.3a: Delivery Status Classification
Multi-class Logistic Regression for predicting order delivery status.

In [ ]:
with mlflow.start_run(run_name="Order_Risk_Classification_Delivery"):
    clf_delivery = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ])
    clf_delivery.fit(X, y_delivery)
    y_pred_delivery = clf_delivery.predict(X)
    acc = accuracy_score(y_delivery, y_pred_delivery)
    
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(clf_delivery, "delivery_classifier")
    
    print(f"\nDelivery Classification Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_delivery, y_pred_delivery, target_names=le_delivery.classes_))
    
    # Confusion Matrix
    cm = confusion_matrix(y_delivery, y_pred_delivery)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le_delivery.classes_, yticklabels=le_delivery.classes_)
    plt.title('Confusion Matrix: Delivery Status Classifier', fontsize=14)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig('../report_screenshots/4_confusion_matrix_delivery.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    joblib.dump(clf_delivery, '../artifacts/delivery_status_model.joblib')
    print("Model saved.")

## 4. Task 3.3b: Customer Segment Classification

In [ ]:
with mlflow.start_run(run_name="Customer_Segment_Classification"):
    clf_segment = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ])
    clf_segment.fit(X, y_segment)
    y_pred_segment = clf_segment.predict(X)
    acc_seg = accuracy_score(y_segment, y_pred_segment)
    
    mlflow.log_metric("accuracy", acc_seg)
    mlflow.sklearn.log_model(clf_segment, "segment_classifier")
    
    print(f"\nSegment Classification Accuracy: {acc_seg:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_segment, y_pred_segment, target_names=le_segment.classes_))
    
    # Confusion Matrix
    cm_seg = confusion_matrix(y_segment, y_pred_segment)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_seg, annot=True, fmt='d', cmap='Greens',
                xticklabels=le_segment.classes_, yticklabels=le_segment.classes_)
    plt.title('Confusion Matrix: Customer Segment Classifier', fontsize=14)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig('../report_screenshots/4b_confusion_matrix_segment.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    joblib.dump(clf_segment, '../artifacts/customer_segment_model.joblib')
    print("Model saved.")